# 06 · Evaluación comparativa: ResNet-50 vs ViT-B/16 (+ O-HAZE)

**Camanchaca-Predict — G5 · Proyecto Aplicado 2026-2**

Notebook de cierre: produce **todas las tablas y figuras para la
presentación**.

1. Carga ambos checkpoints y evalúa en el MISMO test set.
2. Tabla comparativa: MAE / RMSE / MAPE / R² / F1 macro / **falso-seguro** /
   latencia en T4.
3. Figuras: scatter pred-vs-real (log-log), error por banda, matrices de
   confusión, barras comparativas.
4. **Test cualitativo O-HAZE** (niebla real): predicciones sin etiqueta —
   evidencia de generalización (riesgo R1).
5. Exporta `reports/` (figuras + `resumen_para_slides.md`).

Requiere: notebooks 04 y 05 entrenados (checkpoints en Drive).

In [ ]:
# Setup estándar del proyecto
import json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

try:
    import timm
except ImportError:
    !pip install -q timm
    import timm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/camanchaca")
except ImportError:
    ROOT = Path("local_workspace")

DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "figures"
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"Dispositivo: {DEVICE}")

In [ ]:
# Reconstruimos ambos modelos (idénticos a notebooks 04/05) y cargamos pesos
class ResNetVisibility(nn.Module):
    def __init__(self):
        super().__init__()
        net = models.resnet50(weights=None)
        net.fc = nn.Identity()
        self.backbone = net
        self.head = nn.Sequential(nn.Linear(2048, 256), nn.ReLU(inplace=True),
                                  nn.Dropout(0.2), nn.Linear(256, 1))
    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(-1)

class ViTVisibility(nn.Module):
    def __init__(self, n_bands=4):
        super().__init__()
        self.trunk = timm.create_model(
            "vit_base_patch16_224.augreg_in21k_ft_in1k", pretrained=False,
            num_classes=0)
        d = self.trunk.num_features
        self.head_reg = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1))
        self.head_cls = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, n_bands))
    def forward(self, x):
        f = self.trunk(x)
        return self.head_reg(f).squeeze(-1), self.head_cls(f)

resnet = ResNetVisibility().to(DEVICE)
resnet.load_state_dict(torch.load(CKPT_DIR / "resnet50_best.pt",
                                  map_location=DEVICE)["state_dict"])
vit = ViTVisibility().to(DEVICE)
vit.load_state_dict(torch.load(CKPT_DIR / "vitb16_best.pt",
                               map_location=DEVICE)["state_dict"])
with open(DATA_DIR / "splits" / "class_weights.json") as f:
    stats = json.load(f)
MU, SIGMA = stats["mu"], stats["sigma"]
print("Checkpoints cargados ✅")

In [ ]:
# Test set + loaders (idénticos a los notebooks de entrenamiento)
test_df = pd.read_csv(DATA_DIR / "splits" / "test.csv")
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class VisibilityDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = self.transform(Image.open(r["path"]).convert("RGB"))
        return (img, float(r["y_norm"]), int(r["band_idx"]), float(r["visibility_m"]))

test_loader = DataLoader(VisibilityDataset(test_df, eval_tf), batch_size=64,
                         num_workers=2, pin_memory=True)
print(f"test: {len(test_df)} imágenes")

In [ ]:
# Predicciones de ambos modelos sobre el mismo test
@torch.no_grad()
def predict(model, loader, dual=False):
    model.eval()
    preds, vtrues = [], []
    for x, y_norm, band, v_true in loader:
        x = x.to(DEVICE, non_blocking=True)
        out = model(x)
        reg = out[0] if dual else out
        preds.append(reg.float().cpu().numpy())
        vtrues.append(v_true.numpy())
    return np.concatenate(preds), np.concatenate(vtrues)

y_rn, v_true = predict(resnet, test_loader, dual=False)
y_vt, _ = predict(vit, test_loader, dual=True)

def to_meters(y_norm):
    return 10 ** (y_norm * SIGMA + MU)
v_pred_rn = to_meters(y_rn)
v_pred_vt = to_meters(y_vt)
print(f"Predicciones listas: ResNet {len(v_pred_rn)} · ViT {len(v_pred_vt)}")

In [ ]:
# Métricas + latencia (ms/imagen en la GPU actual)
def compute_metrics(v_true, v_pred):
    from sklearn.metrics import f1_score
    err = v_pred - v_true
    mae = float(np.abs(err).mean())
    rmse = float(np.sqrt((err ** 2).mean()))
    mape = float((np.abs(err) / np.clip(v_true, 1e-6, None)).mean() * 100)
    ss_res = float((err ** 2).sum())
    ss_tot = float(((v_true - v_true.mean()) ** 2).sum())
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    f1 = float(f1_score(np.digitize(v_true, (50, 100, 200)),
                        np.digitize(v_pred, (50, 100, 200)),
                        average="macro", zero_division=0))
    peligro = v_true < 100.0
    fs = float((v_pred[peligro] >= 100.0).mean()) if peligro.any() else 0.0
    return {"mae_m": mae, "rmse_m": rmse, "mape_pct": mape, "r2": r2,
            "f1_macro": f1, "false_safe_rate": fs}

def latency_ms(model, dual=False, n_warmup=10, n_iter=50):
    model.eval()
    dummy = torch.randn(1, 3, 224, 224, device=DEVICE)
    with torch.no_grad():
        for _ in range(n_warmup):
            model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_iter):
            model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_iter * 1000

m_rn = compute_metrics(v_true, v_pred_rn)
m_vt = compute_metrics(v_true, v_pred_vt)
m_rn.update({"model": "ResNet-50 (baseline)", "latency_ms": round(latency_ms(resnet), 1)})
m_vt.update({"model": "ViT-B/16 (protagonista)", "latency_ms": round(latency_ms(vit, dual=True), 1)})

tabla = pd.DataFrame([m_rn, m_vt]).set_index("model")
tabla[["mae_m", "rmse_m", "mape_pct", "r2", "f1_macro",
       "false_safe_rate", "latency_ms"]].round(3)

In [ ]:
# FIGURA (slides): scatter pred vs real, log-log, ambos modelos
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), constrained_layout=True)
for ax, (nombre, vp) in zip(axes, [("ResNet-50", v_pred_rn), ("ViT-B/16", v_pred_vt)]):
    ax.scatter(v_true, vp, s=10, alpha=0.45, color="#3681a6", edgecolors="none")
    lims = [max(5, v_true.min()), max(v_true.max(), vp.max())]
    ax.plot(lims, lims, "r--", lw=1.2, label="y = x")
    for e in (50, 100, 200):
        ax.axvline(e, color="gray", ls=":", lw=0.8)
        ax.axhline(e, color="gray", ls=":", lw=0.8)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Visibilidad real [m]"); ax.set_ylabel("Visibilidad predicha [m]")
    ax.set_title(nombre); ax.legend()
fig.suptitle("Test: predicción vs realidad (líneas grises = cortes de banda)")
plt.savefig(FIG_DIR / "scatter_pred_vs_true.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# FIGURA (slides): error absoluto por banda real
BAND_NAMES = ("critico", "alto_riesgo", "precaucion", "aceptable")
b_true = np.digitize(v_true, (50, 100, 200))
err_rn = np.abs(v_pred_rn - v_true)
err_vt = np.abs(v_pred_vt - v_true)

fig, ax = plt.subplots(figsize=(8, 4.2), constrained_layout=True)
data_rn = [err_rn[b_true == i] for i in range(4)]
data_vt = [err_vt[b_true == i] for i in range(4)]
pos = np.arange(4)
bp1 = ax.boxplot(data_rn, positions=pos - 0.19, widths=0.34, patch_artist=True,
                 medianprops=dict(color="k"))
bp2 = ax.boxplot(data_vt, positions=pos + 0.19, widths=0.34, patch_artist=True,
                 medianprops=dict(color="k"))
for b in bp1["boxes"]:
    b.set_facecolor("#67a9cf")
for b in bp2["boxes"]:
    b.set_facecolor("#2166ac")
ax.set_xticks(pos); ax.set_xticklabels(BAND_NAMES)
ax.set_ylabel("Error absoluto [m]")
ax.legend([bp1["boxes"][0], bp2["boxes"][0]], ["ResNet-50", "ViT-B/16"])
ax.set_title("Error absoluto por banda real (test)")
plt.savefig(FIG_DIR / "error_por_banda.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# FIGURA (slides): matrices de confusión lado a lado
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), constrained_layout=True)
for ax, (nombre, vp) in zip(axes, [("ResNet-50", v_pred_rn), ("ViT-B/16", v_pred_vt)]):
    cm = confusion_matrix(b_true, np.digitize(vp, (50, 100, 200)), labels=[0, 1, 2, 3])
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(BAND_NAMES, rotation=25, ha="right")
    ax.set_yticklabels(BAND_NAMES)
    ax.set_xlabel("Predicha"); ax.set_ylabel("Real"); ax.set_title(nombre)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black")
plt.savefig(FIG_DIR / "confusion_comparada.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# FIGURA (slides): barras comparativas (MAE y falso-seguro)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
nombres = ["ResNet-50", "ViT-B/16"]
axes[0].bar(nombres, [m_rn["mae_m"], m_vt["mae_m"]],
            color=["#67a9cf", "#2166ac"], width=0.55)
axes[0].set_ylabel("MAE (m)"); axes[0].set_title("Error absoluto medio")
for i, v in enumerate([m_rn["mae_m"], m_vt["mae_m"]]):
    axes[0].text(i, v, f"{v:.1f} m", ha="center", va="bottom")

axes[1].bar(nombres, [m_rn["false_safe_rate"], m_vt["false_safe_rate"]],
            color=["#67a9cf", "#2166ac"], width=0.55)
axes[1].set_ylabel("tasa de falso-seguro"); axes[1].set_title("V real < 100 m predicha ≥ 100 m")
for i, v in enumerate([m_rn["false_safe_rate"], m_vt["false_safe_rate"]]):
    axes[1].text(i, v, f"{v:.1%}", ha="center", va="bottom")
plt.savefig(FIG_DIR / "comparativa_barras.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Ejemplos cualitativos: mejores y peores predicciones del ViT
err_vt_abs = err_vt
orden = np.argsort(err_vt_abs)
mejores, peores = orden[:4], orden[-4:]

fig, axes = plt.subplots(2, 4, figsize=(14, 6.6), constrained_layout=True)
for fila, (idxs, titulo) in enumerate([(mejores, "Mejores predicciones"),
                                       (peores, "Peores predicciones")]):
    for k, idx in enumerate(idxs):
        ax = axes[fila, k]
        r = test_df.iloc[idx]
        ax.imshow(Image.open(r["path"]))
        ax.axis("off")
        ax.set_title(f"real {v_true[idx]:.0f} m · ViT {v_pred_vt[idx]:.0f} m\n"
                     f"({BAND_NAMES[b_true[idx]]})", fontsize=9)
    axes[fila, 0].text(-0.06, 0.5, titulo, transform=axes[fila, 0].transAxes,
                       rotation=90, va="center", ha="center", fontsize=11)
plt.savefig(FIG_DIR / "cualitativos_vit.png", dpi=150, bbox_inches="tight")
plt.show()

## Test cualitativo O-HAZE (niebla REAL) — riesgo R1

O-HAZE no tiene etiqueta en metros: aquí **no medimos, inspeccionamos**. Si
los modelos predicen visibilidades plausibles (bajas) y ordenan las escenas
de forma consistente con su densidad aparente de niebla, hay evidencia de
generalización síntesis → real. Si no, documentamos la brecha (honestidad
científica → `docs/risks_and_mitigation.md` R1).

> O-HAZE se solicita con correo institucional en la página del NTIRE 2018
> Image Dehazing Challenge y se organiza con
> `python scripts/download_ohaze.py --zip O-HAZE.zip`
> (carpeta esperada: `data/raw/ohaze/hazy/`).

In [ ]:
# O-HAZE (condicional): inferencia cualitativa sin etiqueta
OHAZE_DIR = DATA_DIR / "raw" / "ohaze" / "hazy"
hazy_paths = sorted(OHAZE_DIR.glob("*.png")) + sorted(OHAZE_DIR.glob("*.jpg")) \
    if OHAZE_DIR.exists() else []

if not hazy_paths:
    print("O-HAZE no disponible — el test cualitativo se omite.")
    print("Instrucciones en la celda markdown anterior y en scripts/download_ohaze.py")
else:
    @torch.no_grad()
    def predict_single(model, pil_img, dual=False):
        x = eval_tf(pil_img.convert("RGB")).unsqueeze(0).to(DEVICE)
        out = model(x)
        reg = out[0] if dual else out
        return float(to_meters(reg.float().cpu().numpy()[0]))

    sel = hazy_paths[:8]
    fig, axes = plt.subplots(2, 4, figsize=(14, 6.6), constrained_layout=True)
    for k, p in enumerate(sel):
        ax = axes[k // 4, k % 4]
        img = Image.open(p)
        vr = predict_single(resnet, img)
        vv = predict_single(vit, img, dual=True)
        banda = BAND_NAMES[min(3, np.digitize(vv, (50, 100, 200)))]
        ax.imshow(img); ax.axis("off")
        ax.set_title(f"ResNet {vr:.0f} m · ViT {vv:.0f} m\n({banda})",
                     fontsize=9, color="#8b0000")
    fig.suptitle("O-HAZE (niebla REAL, sin etiqueta): predicciones de visibilidad", fontsize=13)
    plt.savefig(FIG_DIR / "ohaze_cualitativo.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Exportamos TODO a reports/ (repo) + resumen para los slides
import shutil

REPO_LOCAL = Path("/content/camanchaca-predict")
figuras = ["scatter_pred_vs_true.png", "error_por_banda.png",
           "confusion_comparada.png", "comparativa_barras.png",
           "cualitativos_vit.png"]
if OHAZE_DIR.exists() and hazy_paths:
    figuras.append("ohaze_cualitativo.png")

filas = ["| Modelo | MAE (m) | RMSE (m) | MAPE | R2 | F1 macro | Falso-seguro | Latencia (ms) |",
         "|---|---|---|---|---|---|---|---|"]
for m in (m_rn, m_vt):
    filas.append(f"| {m['model']} | {m['mae_m']:.1f} | {m['rmse_m']:.1f} | "
                 f"{m['mape_pct']:.1f}% | {m['r2']:.3f} | {m['f1_macro']:.3f} | "
                 f"{m['false_safe_rate']:.1%} | {m['latency_ms']:.1f} |")
resumen = "# Resultados para la presentación (generado por el notebook 06)\n\n" + "\n".join(filas) + "\n"
(RESULTS_DIR / "resumen_para_slides.md").write_text(resumen, encoding="utf-8")
print(resumen)

if REPO_LOCAL.exists():
    figdir = REPO_LOCAL / "reports" / "figures"
    figdir.mkdir(parents=True, exist_ok=True)
    for f in figuras:
        if (FIG_DIR / f).exists():
            shutil.copy(FIG_DIR / f, figdir / f)
    shutil.copy(RESULTS_DIR / "resumen_para_slides.md",
                REPO_LOCAL / "reports" / "tables" / "resumen_para_slides.md")
    print(f"Figuras copiadas al repo: {len(figuras)}")

with open(RESULTS_DIR / "comparison.json", "w") as f:
    json.dump({"resnet": m_rn, "vit": m_vt}, f, indent=2)
print("comparison.json guardado ✅")

## Conclusiones (plantilla para completar tras el run)

1. **Comparación:** ¿el ViT superó al ResNet-50 en MAE y falso-seguro?
   ¿A qué costo (parámetros ×3.5, latencia ×?)?
2. **Seguridad:** la tasa de falso-seguro es la métrica de despliegue; un
   sistema real exigiría umbral conservador (predicción mínima de un ensemble).
3. **Generalización (O-HAZE):** comportamiento cualitativo en niebla real y
   brecha documentada (R1).
4. **Camanchaca real:** sin datos chilenos en esta fase — roadmap de capturas
   propias/cámaras DTV para fine-tuning local.

Con estas figuras y `resumen_para_slides.md` tienes TODO el material
cuantitativo de la presentación. El deck sigue el guion de
`docs/avance_2026-09-21.md`.